In [ ]:
# In Jupyter, run this to force restart
import os
os._exit(00)

In [3]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python path: {sys.path}")

Python executable: c:\Users\Anupam\Desktop\LLM\.venv\Scripts\python.exe
Python path: ['C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10\\python310.zip', 'C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10\\DLLs', 'C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10\\lib', 'C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv', '', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages\\win32', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages\\win32\\lib', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages\\Pythonwin']


In [4]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.9.1+cu126
True


In [6]:
import subprocess
import json

def compare_with_requirements():
    """Compare current versions with old requirements files"""
    
    # Get current versions
    result = subprocess.run(
        [sys.executable, "-m", "pip", "list", "--format=json"],
        capture_output=True, text=True
    )
    current = {pkg['name']: pkg['version'] for pkg in json.loads(result.stdout)}
    
    # Check for old requirements files
    req_files = ['requirements.txt', 'requirements-old.txt', 
                 'requirements-backup.txt', '../requirements.txt']
    
    for req_file in req_files:
        if os.path.exists(req_file):
            print(f"\n📄 Comparing with {req_file}:")
            with open(req_file, 'r') as f:
                for line in f:
                    if '==' in line:
                        pkg, ver = line.strip().split('==')
                        pkg = pkg.lower()
                        if pkg in current:
                            if current[pkg] != ver:
                                print(f"  {pkg}: {ver} → {current[pkg]} (changed)")
                            else:
                                print(f"  {pkg}: {ver} (unchanged)")

In [5]:
import packaging
print(f"packaging: {packaging.__version__}")

packaging: 26.0


In [6]:
import os
import torch
from dotenv import load_dotenv
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl.trainer.sft_trainer import SFTTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0316 13:21:07.312000 23108 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
load_dotenv()  # Load environment variables from .env file
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN") or ""

In [8]:
# Disable torch.compile for stability
# Set TORCH_COMPILE_DISABLE=1 and torch_compile=False
torch._dynamo.config.disable = True
os.environ["TORCH_COMPILE_DISABLE"] = "1"

Mlflow implementation

In [14]:
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")  # Set the tracking URI to your MLflow server
mlflow.set_experiment("LLM_Fine_Tuning")  # Set the experiment name

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1773153482475, experiment_id='1', last_update_time=1773153482475, lifecycle_stage='active', name='LLM_Fine_Tuning', tags={}, workspace='default'>

Load the base model and tokenizer

In [15]:
max_seq_length = 2048
model_name= "unsloth/Qwen3.5-2B"
# Processor should be fetch where multimodal special tokens are present
model, processor = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit = False,     # MoE QLoRA not recommended, dense 27B is fine
    load_in_16bit = True,     # bf16/16-bit LoRA
    full_finetuning = False,
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Current model requires 320 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
Loading weights: 100%|██████████| 617/617 [00:00<00:00, 692.06it/s, Materializing param=model.visual.pos_embed.weight]                                 
[accelerate.big_modeling|WARNING]Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [ ]:
# 2. CRITICAL: Ensure model is fully on GPU before loading adapter
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)  # Force to GPU

In [16]:
type(processor.tokenizer)
print(processor.tokenizer)
# CRITICAL: Extract the text tokenizer from the vision processor
tokenizer = processor.tokenizer  # This gives you the text-only tokenizer
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
print(f"Set pad_token to: {tokenizer.pad_token}")
print(f"Vocab size: {len(tokenizer)}")


TokenizersBackend(name_or_path='unsloth/Qwen3.5-2B', vocab_size=248044, model_max_length=262144, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|vision_pad|>', 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'vision_bos_token': '<|vision_start|>', 'vision_eos_token': '<|vision_end|>'}, added_tokens_decoder={
	248044: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248045: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248046: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248047: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248048: AddedToken("<|object_ref_end|>"

In [17]:
tokenizer.tokenize

<bound method TokenizersBackend.tokenize of TokenizersBackend(name_or_path='unsloth/Qwen3.5-2B', vocab_size=248044, model_max_length=262144, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|im_end|>', 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'vision_bos_token': '<|vision_start|>', 'vision_eos_token': '<|vision_end|>'}, added_tokens_decoder={
	248044: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248045: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248046: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248047: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),


Apply LoRA adapters for efficient fine-tuning

In [18]:
# Enable peft for memory efficient training
# Disable vision fine-tuning - only train text layers
# This effectively treats it as a text-only model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",  # Text layers only
    ],
    lora_alpha = 16,
    # without dropout and without bias, to speed up training.
    lora_dropout = 0,
    bias = "none",
    # "unsloth" checkpointing is intended for very long context + lower VRAM
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = max_seq_length,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


Load and format the training dataset

In [19]:
# Load the dataset from huggingface
dataset = load_dataset("nohurry/Opus-4.6-Reasoning-3000x-filtered", split="train")
print(f"Original columns name:{dataset.column_names}")
# Keep only the columns you want (e.g., 'conversations' and 'text')
columns_to_keep = ['problem','thinking', 'solution']  # Replace with your actual column names
dataset = dataset.select_columns(columns_to_keep)
print(f"New columns: {dataset.column_names}")
# dataset = dataset.train_test_split(test_size=0.1, seed=3407)

Original columns name:['id', 'problem', 'thinking', 'solution', 'difficulty', 'category', 'timestamp', 'hash']
New columns: ['problem', 'thinking', 'solution']


In [20]:
type(dataset[0])
dataset[0]

{'problem': "252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?",
 'thinking': "Let me work through this problem step by step.\n\nFirst, I need to understand what's being asked: 252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?\n\nKey values given: 252, 8, 41, 300,000, 7,500, ,\n\nMy approach:\n1. Find the total number of people going on the field trip\n\n- Fifth-grade students: 252\n- Teachers: 8\n- Total people: 252 + 8 = 260 people\n2. Calculate how many buses are needed\n\nEach bus has 41 seats.\n\n$$\\text{Number of buses} = \\frac{260}{41} = 6.34...$$\n\nSince we cannot rent a partial bus, we must round up to the\

In [21]:
# 2. Format into conversations
def format_reasoning_conversation(examples):
    conversations = []
    for i in range(len(examples['problem'])):
        conversations.append([
            {"role": "user", "content": examples['problem'][i]},
            {"role": "assistant", "content": f"Let me think through this step-by-step:\n\n{examples['thinking'][i]}\n\nTherefore, the solution is:\n\n{examples['solution'][i]}"}
        ])
    return {"conversations": conversations}

print(f"Original dataset before the mapping:{dataset}")
dataset = dataset.map(format_reasoning_conversation, batched=True, remove_columns=dataset.column_names)
print(f"New dataset columns after the removing of previous columns:{dataset}")

Original dataset before the mapping:Dataset({
    features: ['problem', 'thinking', 'solution'],
    num_rows: 2326
})
New dataset columns after the removing of previous columns:Dataset({
    features: ['conversations'],
    num_rows: 2326
})


In [22]:
dataset["conversations"][0]

[{'content': "252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?",
  'role': 'user'},
 {'content': "Let me think through this step-by-step:\n\nLet me work through this problem step by step.\n\nFirst, I need to understand what's being asked: 252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?\n\nKey values given: 252, 8, 41, 300,000, 7,500, ,\n\nMy approach:\n1. Find the total number of people going on the field trip\n\n- Fifth-grade students: 252\n- Teachers: 8\n- Total people: 252 + 8 = 260 people\n2. Calculate how many buses are needed\n\nEach bus has 41 seats.\n\n$$\\text{Number of buses} = \\frac{260}{41} = 6.34...$$\

Apply the Qwen3.5:2b chat template

In [23]:
# For the ChatML format
chat_template= """
<|im_start|>user
{user}<|im_end|>
<|im_start|>assistant
{OUTPUT}<|im_end|>"""

In [24]:
from unsloth import get_chat_template, standardize_sharegpt
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml"
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False,add_generation_prompt=False) for convo in convos
    ]
    return {
        "text": texts
    }

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


Model does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [25]:
print(tokenizer.pad_token)

<|PAD_TOKEN|>


In [26]:
dataset

Dataset({
    features: ['conversations'],
    num_rows: 2326
})

In [27]:
dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)

In [28]:
print(type(dataset))
print(dataset["text"][0])
print(dataset.column_names)

<class 'datasets.arrow_dataset.Dataset'>
<|im_start|>user
252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?<|im_end|>
<|im_start|>assistant
Let me think through this step-by-step:

Let me work through this problem step by step.

First, I need to understand what's being asked: 252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?

Key values given: 252, 8, 41, 300,000, 7,500, ,

My approach:
1. Find the total number of people going on the field trip

- Fifth-grade students: 252
- Teachers: 8
- Total people: 252 + 8 = 260 people
2. Calculate how many buses are needed

Each bus has 41 seats.

$$\text{Number of buses} = \frac{

In [29]:
# Split the dataset into train and evaluate set 
dataset = dataset.train_test_split(test_size=0.1, seed=3407)


In [30]:
# Assign train and eval dataset accordingly
train_dataset = dataset["train"]
eval_dataset = dataset['test']

Standardize the conversation format(Optional)

In [31]:
# from unsloth.chat_templates import standardize_sharegpt
# dataset = standardize_sharegpt(dataset)

In [32]:
# from unsloth import is_bfloat16_supported
from trl.trainer.sft_config import SFTConfig

In [33]:
# Define SFT-specific configuration
sft_config = SFTConfig(
        # Training params
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 5,
        warmup_steps = 5,
        max_steps = 120,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        load_best_model_at_end=True,
        
        # Evaluation params (CRITICAL)
        eval_strategy="steps",  # When to evaluate: "steps", "epoch", or "no"
        eval_steps= 20, # Evaluate every 20 steps (if strategy="steps")
        eval_accumulation_steps=1,  # Gradient accumulation for eval (usually 1)
        
        # Logging 
        output_dir = "outputs",
        # CRITICAL: Disable default MLflow reporting to avoid duplicate runs
        report_to = "none",     # For Weights and Biases
        
        # Dataset format (MUST match train)
        # SFTTrainer expects either pre-tokenized data or raw text with single
        # column. Multiple columns ['text', 'input', 'output'] confuse auto-detection.
        remove_unused_columns = False,
        dataset_text_field = "text",
        packing=False, # Must be False for raw text. Can make training 5x faster for short sequences.
        # dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    )

In [34]:
# # Training arguments remain the same
# training_args = TrainingArguments(
#     learning_rate=3e-4,
#     lr_scheduler_type="linear",
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     num_train_epochs=3,
#     fp16=not is_bfloat16_supported(),
#     bf16 = is_bfloat16_supported(),
#     logging_steps=1,
#     optim="adamw_8bit",
#     weight_decay=0.01,
#     warmup_steps=10,
#     output_dir="output",
#     report_to="mlflow",  # This should work with comet_ml installed [citation:4][citation:8]
#     seed=0,
# )

In [35]:
print(tokenizer.eos_token)
print(f"Tokenizer vocab size: {len(tokenizer)}")

<|im_end|>
Tokenizer vocab size: 248078


Check for EOS_TOKEN in tokenizer

In [36]:
print('<|im_end|>' in tokenizer.get_vocab().keys())

True


In [37]:
print('eos_token' in tokenizer.special_tokens_map)

True


In [38]:
# Check what the eos_token actually is
print(f"eos_token string: '{tokenizer.eos_token}'")
print(f"eos_token exists as key in vocab: {tokenizer.eos_token in tokenizer.get_vocab()}")

# Check if it's in the special tokens list instead
print(f"Special tokens: {tokenizer.special_tokens_map}")
print(f"All special tokens: {tokenizer.all_special_tokens}")

# Get the actual token ID
print(f"eos_token_id: {tokenizer.eos_token_id}")

# See what token is at that ID (reverse lookup)
if tokenizer.eos_token_id is not None:
    token_at_eos_id = tokenizer.convert_ids_to_tokens(tokenizer.eos_token_id)
    print(f"Token at eos_token_id ({tokenizer.eos_token_id}): '{token_at_eos_id}'")

eos_token string: '<|im_end|>'
eos_token exists as key in vocab: True
Special tokens: {'eos_token': '<|im_end|>', 'pad_token': '<|PAD_TOKEN|>'}
All special tokens: ['<|im_end|>', '<|PAD_TOKEN|>']
eos_token_id: 248046
Token at eos_token_id (248046): '<|im_end|>'


In [39]:
# Initialize SFTTrainer with proper parameter placement
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,  # Correct for v0.22.0 [citation:5]
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config,            # Pass SFTConfig here [citation:9]
)

In [40]:
# Create a custom callback for precise metric logging (optional)

from transformers import TrainerCallback
class PreciseMetricsCallback(TrainerCallback):
    def __init__(self, run):
        self.run = run
        self._run_active = False
        self.train_history = []
        self.eval_history = []
        
    def on_train_begin(self, args, state, control, **kwargs):
        """Initialize MLflow run if not already active."""
        if not mlflow.active_run():
            self.run_id = self.run.info.run_id
            self._run_active= True
            # Log all training hyperparameters
            mlflow.log_params({
                "learning_rate": args.learning_rate,
                "batch_size": args.per_device_train_batch_size,
                "gradient_accumulation_steps": args.gradient_accumulation_steps,
                "warmup_steps": args.warmup_steps,
                "max_steps": args.max_steps,
                "weight_decay": args.weight_decay,
                "total_train_batch_size": args.per_device_train_batch_size * 
                                         args.gradient_accumulation_steps * 
                                         args.world_size,
            })
            
        print(f"✓ MLflow logging to run: {mlflow.active_run().info.run_id}")            
        
    def on_log(self, args, state, control, logs= None, **kwargs):
        """Log training metrics with 'train/' prefix."""
        if not logs:
            return
        
        train_metrics = {}
        for k, v in logs.items():
            if isinstance(v, (int,float)):
                if k in ["eval_loss", "eval_runtime","eval_samples_per_second"]:
                    continue
                train_metrics[f"train/{k}"]=v
                
                
        if train_metrics:
            step = int(state.global_step)
            mlflow.log_metrics(train_metrics, step = step)
            
            # store more comparison
            self.train_history.append({"step":step, **train_metrics})
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        """
        Log evaluation metrics with 'eval/' prefix.
        """
        # Hugging Face usually passes metrics directly; if not, check kwargs
        if metrics is None:
            metrics = kwargs.get("metrics", {})
        
        if not metrics:
            print("⚠ No metrics provided to on_evaluate")
            return
            
        step = int(state.global_step)
        
        # Prefix all eval metrics and ensure they are numeric
        eval_metrics = {
            f"eval/{k}": v for k, v in metrics.items() 
            if isinstance(v, (int, float)) and not k.startswith("eval/")
        }
        
        # Log to MLflow
        if eval_metrics:
            mlflow.log_metrics(eval_metrics, step=step)
        
        # Store for comparison
        self.eval_history.append({"step": step, **eval_metrics})
        
        # --- Overfitting Monitoring ---
        eval_loss = eval_metrics.get("eval/loss")
        
        if eval_loss and self.train_history:
            # Get the most recent training loss logged
            latest_train_loss = self.train_history[-1].get("train/loss")
            
            if latest_train_loss and latest_train_loss > 0:
                # Ratio > 1 means Eval Loss is higher than Train Loss (Normal/Overfitting)
                # Ratio < 1 means Train Loss is higher (Underfitting)
                ratio = eval_loss / latest_train_loss
                mlflow.log_metric("monitoring/eval_train_loss_ratio", ratio, step=step)
                
                if ratio > 1.5:
                    mlflow.set_tag("warning", "potential_overfitting")
                    print(f"🚨 Warning: Overfitting detected (Ratio: {ratio:.2f})")
                elif ratio < 0.8:
                    mlflow.set_tag("warning", "potential_underfitting")
        
        print(f"✓ Eval metrics logged at step {step}: {list(eval_metrics.keys())}")
        
        
    def on_train_end(self, args, state, control, **kwargs):
        """Log final summary and artifacts."""
        if self._run_active:
            # Log final statistics
            mlflow.log_metrics({
                "training/total_steps": state.global_step,
                "training/best_metric": state.best_metric if state.best_metric else 0,
            })
            
            # End the run we started
            mlflow.end_run()
            self._run_active = False
        

In [41]:
import mlflow 
import math
with mlflow.start_run(run_name="Qwen3.5:2b Pretraining", log_system_metrics=True) as run:
    # Log key hyperparameter explicitly
    mlflow.log_params({
        "model_name": model_name,
        "max_seq_length": max_seq_length,
        "lora_r": 16,
        "lora_alpha": 16,
        "lora_dropout": 0,
        "lora_bias": "none",
        "target_modules": "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj",
        "train_size": len(dataset['train']),
        "eval_size": len(dataset["test"]),
    }
    )
    # Add custom callback into trainer instance
    trainer.add_callback(PreciseMetricsCallback(run=run))
    train_result = trainer.train()
    eval_metrics = trainer.evaluate() # In evaluate only metrics are calculated 
    
    # Final metrics logging
    for k, v in train_result.metrics.items():
        if isinstance(v, (int, float)):
            mlflow.log_metrics(f"train_{k}", float(v))
    for k, v in eval_metrics.items():
        if isinstance(v, (int, float)):
            mlflow.log_metric(k, float(v))
    if "eval_loss" in eval_metrics:
        mlflow.log_metric("final_eval_perplexity", math.exp(eval_metrics["eval_loss"]))
        
        
    # Save and log model artifacts (LoRA adapter + tokenizer)
    

2026/03/16 01:12:48 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/16 01:12:48 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248077}.
2026/03/16 01:12:48 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/16 01:12:48 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run Qwen3.5:2b Pretraining at: http://localhost:5000/#/experiments/1/runs/fdf99d8d8c5c42748278c2937a2e8b1b
🧪 View experiment at: http://localhost:5000/#/experiments/1


ValueError: You can't train a model that has been loaded with `device_map='auto'` in any distributed mode. Please rerun your script specifying `--num_processes=1` or by launching with `python {{myscript.py}}`.

In [ ]:
final_dir = "output/final_lora_adapter"
tokenizer.save_pretrained(final_dir)
mlflow.log_artifacts(final_dir, artifact_path="model")
trainer.save_state()
mlflow.log_artifacts("output", artifact_path = "trainer_output")

Saving, loading finetuned models

In [ ]:
model.push_to_hub("Anupammaiti/qwen_lora", token = os.getenv("HF_TOKEN")) # Online saving
tokenizer.push_to_hub("Anupammaiti/qwen_lora", token = os.getenv("HF_TOKEN")) # Online saving

Processing Files (1 / 1): 100%|██████████| 21.9MB / 21.9MB, 45.8kB/s  
New Data Upload: 100%|██████████| 21.9MB / 21.9MB, 45.8kB/s  


Saved model to https://huggingface.co/Anupammaiti/qwen_lora


Processing Files (1 / 1): 100%|██████████| 20.0MB / 20.0MB, 5.46MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Load the adapter for inference

In [9]:
max_seq_length = 2048
model_name= "unsloth/Qwen3.5-2B"
# Processor should be fetch where multimodal special tokens are present
model, processor = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit = False,     # MoE QLoRA not recommended, dense 27B is fine
    load_in_16bit = True,     # bf16/16-bit LoRA
    full_finetuning = False,
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 617/617 [00:13<00:00, 45.69it/s, Materializing param=model.visual.pos_embed.weight]                                 


In [10]:
# 2. CRITICAL: Ensure model is fully on GPU before loading adapter
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)  # Force to GPU

In [11]:
tokenizer = processor.tokenizer  # Get the tokenizer from the model's processor

In [12]:
device = next(model.parameters()).device
print(f"Model loaded on device: {device}")
# To check if ALL parameters are on CUDA:
all_on_gpu = any(p.is_cuda for p in model.parameters())
print(f"Are all weights on GPU? {all_on_gpu}")

Model loaded on device: cuda:0
Are all weights on GPU? True


In [ ]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   4232 MiB |   4232 MiB |   8442 MiB |   4210 MiB |
|       from large pool |   4228 MiB |   4228 MiB |   8438 MiB |   4210 MiB |
|       from small pool |      4 MiB |      4 MiB |      4 MiB |      0 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   4232 MiB |   4232 MiB |   8442 MiB |   4210 MiB |
|       from large pool |   4228 MiB |   4228 MiB |   8438 MiB |

In [13]:
# Path to your saved adapter
adapter_path = "./outputs/checkpoint-120"  # or wherever you saved
# 1. Load base model (same as training)
# 2. Load your fine-tuned adapter
model.load_adapter(adapter_path)
# 3. Enable fast inference (2x speedup)
FastLanguageModel.for_inference(model)
# Verify all on same device
print(f"Model device: {next(model.parameters()).device}")  # should say cuda:0


Loading weights: 100%|██████████| 192/192 [00:00<00:00, 643.28it/s, Materializing param=model.language_model.layers.23.self_attn.v_proj.lora_B.default.weight]


Model device: cuda:0


In [14]:
model.to(device)

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [15]:
for name, param in model.named_parameters():
    print(f"{name}: {param.device}")


model.visual.patch_embed.proj.weight: cuda:0
model.visual.patch_embed.proj.bias: cuda:0
model.visual.pos_embed.weight: cuda:0
model.visual.blocks.0.norm1.weight: cuda:0
model.visual.blocks.0.norm1.bias: cuda:0
model.visual.blocks.0.norm2.weight: cuda:0
model.visual.blocks.0.norm2.bias: cuda:0
model.visual.blocks.0.attn.qkv.weight: cuda:0
model.visual.blocks.0.attn.qkv.bias: cuda:0
model.visual.blocks.0.attn.proj.weight: cuda:0
model.visual.blocks.0.attn.proj.bias: cuda:0
model.visual.blocks.0.mlp.linear_fc1.weight: cuda:0
model.visual.blocks.0.mlp.linear_fc1.bias: cuda:0
model.visual.blocks.0.mlp.linear_fc2.weight: cuda:0
model.visual.blocks.0.mlp.linear_fc2.bias: cuda:0
model.visual.blocks.1.norm1.weight: cuda:0
model.visual.blocks.1.norm1.bias: cuda:0
model.visual.blocks.1.norm2.weight: cuda:0
model.visual.blocks.1.norm2.bias: cuda:0
model.visual.blocks.1.attn.qkv.weight: cuda:0
model.visual.blocks.1.attn.qkv.bias: cuda:0
model.visual.blocks.1.attn.proj.weight: cuda:0
model.visual.bl

In [47]:
# Test the fine-tuned model with a sample prompt
test_prompt = """User : No solution found when resolving dependencies:
  ╰─▶ Because there is no version of torch==2.9.1+cu126 and you require torch==2.9.1+cu126, we can conclude that your requirements are unsatisfiable.
Assistant:"""

In [48]:
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
inputs

{'input_ids': tensor([[  1421,    535,   2233,   6093,   1669,    948,  50716,  18948,     25,
            198,    220,  56252,    108,  14136, 169199,   8938,   1017,    369,
            874,   2243,    314,   7586,    410,     17,     13,     24,     13,
             16,     10,  19586,     16,     17,     21,    321,    488,   1325,
           7586,    410,     17,     13,     24,     13,     16,     10,  19586,
             16,     17,     21,     11,    567,    628,  30076,    421,    678,
           8242,    513,   6758,   7243,   2087,     13,    198,  69267,     25]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [9]:
# inputs = tokenizer.decode(inputs.input_ids[0])
# print(inputs)

In [49]:
import torch
from threading import Thread
from transformers import TextIteratorStreamer
from IPython.display import display, Markdown, HTML

# 1. Setup the Streamer
streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, decode_kwargs={"skip_special_tokens": True})

# 2. Define the Generation in a Thread
# This prevents the UI from locking up
generation_kwargs = dict(
    **inputs, 
    streamer=streamer, 
    max_new_tokens=1024,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

thread = Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

# 3. Create a Smooth Display Handle
# We initialize it with a 'thinking' message or empty string
display_handle = display(Markdown("Thinking..."), display_id=True)

full_text = ""

# 4. Stream Loop
for new_text in streamer:
    full_text += new_text
    
    # Optional: Add a subtle 'cursor' like ChatGPT
    streaming_html = f"{full_text}█"
    
    # Update the existing handle without clearing the whole output
    display_handle.update(Markdown(streaming_html))

# 5. Final Update to remove the cursor once finished
display_handle.update(Markdown(full_text))

 Here is a breakdown of the error you are encountering and how to fix it.

### The Error Analysis
The error `Because there is no version of torch==2.9.1+cu126 and you require torch==2.9.1+cu126, we can conclude that your requirements are unsatisfiable` indicates a **cycle or duplicate constraint** in your `requirements.txt` file.

1.  **The Logic:** Python's dependency resolver tries to satisfy your requirements. When it hits your own file, it sees:
    *   Requirement A: `torch==2.9.1+cu126`
    *   Requirement B: `torch==2.9.1+cu126`
    *   It cannot satisfy both at the same time because only one instance of that specific version can exist in a single package.
2.  **The Result:** The package installer (pip) fails immediately because the constraints create an impossible state.

---

### Solution 1: The "Remove Duplicate" Fix (Recommended)
Because `torch` usually only needs one version in the base environment, adding the same version twice is almost certainly a mistake.

1.  Open your `requirements.txt` file in a text editor.
2.  Find the line(s) that contain `torch==2.9.1+cu126`.
3.  **Delete the duplicate lines** or replace the second line with nothing (or a different requirement).

**File structure should look like this (Top level):**
```text
# This file lists the dependencies required for your project.
# It does not include the base system libraries (python, pip, numpy).

torch==2.9.1+cu126
# (Optional) numpy==1.26.4
# (Optional) torchvision==0.9.1+cu126 -- extra-index-url=https://download.pytorch.org/whl/cu121
```

**If the file looks more complex (e.g., using extras):**
```text
torch==2.9.1+cu126

[extra1]
torch==2.9.1+cu126

[extra2]
torch==2.9.1+cu126
```
*In this case, you likely only need one version of torch. To fix this, remove the lines under `[extra1]` and `[extra2]`, or ensure the extras aren't conflicting.*

---

### Solution 2: The "Fix" Fix (If you need multiple versions)
If you genuinely need two different versions of `torch` (perhaps for testing or specific architectures) and you **must** have them both, you need to explicitly declare that they are separate "groups" or "environments" rather than conflicting requirements.

You can use the `[extra]` syntax to separate them:

**File structure:**
```text
torch==2.9.1+cu126
torch==2.9.2+cu126

[extra-1]
# This group only contains version 2.9.1
torch==2.9.1+cu126

[extra-2]
# This group only contains version 2.9.2
torch==2.9.2+cu126
```

**How to install:**
Run this command in your terminal:
```bash
pip install -r requirements.txt -f https://download.pytorch.org/whl/torch_stable.html
```
*Note: The `extra-index-url` is no longer strictly necessary if you are using the `[extra]` syntax, but it helps if you need to access specific wheels.*

### Summary
In 99% of cases, **Solution 1** is the correct fix. You simply need to ensure that `torch==2.9.1+cu126` appears only **once** in your `requirements.txt`. Once you remove the duplicate, the error will disappear, and the installation will continue.<|endoftext|>